In [1]:
import sys
sys.path.append('../')

%env MUJOCO_GL=egl

env: MUJOCO_GL=egl


In [13]:
import jax
import jax.numpy as jnp
import mujoco
import numpy as np
import mediapy as media
from dataclasses import dataclass, field
from mujoco.mjx._src import math as mjx_math

from builderbench.env_utils import make_env
from utils.wrapper import wrap_env

In [128]:
@dataclass
class Args:
    # experiment
    agent: str = "mp"
    seed: int = 1

    # environment
    env_id: str = 'creative-1-task1'
    env_early_termination: bool = True
    env_episode_length: int = None
    permutation_invariant_reward: bool = True   # invariance to the order of cubes in any structure

    # planner
    kp_pos: float = 10.0
    kd_pos: float = 2.0

    kp_yaw: float = 10.0
    kd_yaw: float = 0.5
    
    num_envs: str = 1
    

In [129]:
args = Args()

In [130]:
env_class, default_config = make_env(args)
env = env_class(config=default_config)

In [131]:
np.random.seed(args.seed)
key = jax.random.PRNGKey(args.seed)
key, key_env, key_eval, key_policy, key_value = jax.random.split(key, 5)

In [132]:
reset_fn = jax.jit(env.reset)
step_fn = jax.jit(env.step)

In [158]:
@jax.jit
def get_yaw_from_quat(q):
    w, x, y, z = q[0], q[1], q[2], q[3]
    siny_cosp = 2 * (w * z + x * y)
    cosy_cosp = 1 - 2 * (y * y + z * z)
    yaw = jnp.arctan2(siny_cosp, cosy_cosp)
    return yaw

@jax.jit
def normalize_angle(angle):
    return jnp.arctan2(jnp.sin(angle), jnp.cos(angle))

In [159]:
@jax.jit
def get_waypoint(env_state):
    current_pos = env_state.obs[:3*env._config.num_cubes].reshape(env._config.num_cubes, 3)[cube_id]
    current_quat = env_state.obs[3*env._config.num_cubes:][:4*env._config.num_cubes].reshape(env._config.num_cubes, 4)[cube_id]
    current_linvel = env_state.obs[7*env._config.num_cubes:][:3*env._config.num_cubes].reshape(env._config.num_cubes, 3)[cube_id]
    current_angvel = env_state.obs[10*env._config.num_cubes:][:3*env._config.num_cubes].reshape(env._config.num_cubes, 3)[cube_id]
    
    target_pos = env_state.info['target_goal'].reshape(env._config.num_cubes, 3)[cube_id]
    
    current_xy = current_pos[:2]
    target_xy = target_pos[:2]
    
    current_height = current_pos[-1]
    top_height = target_pos[-1] + 0.1
    
    horizontal_dist_to_target = jnp.linalg.norm(current_xy - target_xy)

    is_far = horizontal_dist_to_target > 0.01
    is_low = current_height < (top_height - 0.005)

    wp_lift = target_pos.at[:2].set(current_xy).at[2].set(top_height)
    wp_hover = target_pos.at[2].set(top_height)
    wp_final = target_pos
        
    current_waypoint = jnp.where(
        is_far,
        jnp.where(is_low, wp_lift, wp_hover),
        wp_final
    )

    return current_waypoint

@jax.jit
def get_action(env_state, waypoint, cube_id):
    current_pos = env_state.obs[:3*env._config.num_cubes].reshape(env._config.num_cubes, 3)[cube_id]
    current_quat = env_state.obs[3*env._config.num_cubes:][:4*env._config.num_cubes].reshape(env._config.num_cubes, 4)[cube_id]
    current_linvel = env_state.obs[7*env._config.num_cubes:][:3*env._config.num_cubes].reshape(env._config.num_cubes, 3)[cube_id]
    current_angvel = env_state.obs[10*env._config.num_cubes:][:3*env._config.num_cubes].reshape(env._config.num_cubes, 3)[cube_id]
    
    current_yaw = get_yaw_from_quat(current_quat)
    
    error_pos = waypoint - current_pos
    output_pos = (args.kp_pos * error_pos) + (args.kd_pos * - current_linvel)
    output_pos = output_pos.at[2].add(gravity_comp)
    
    error_yaw = normalize_angle(0.0 - current_yaw)
    output_yaw = (args.kp_yaw * error_yaw) + (args.kd_yaw * - current_angvel[-1])

    raw_ctrl_action = jnp.concatenate([output_pos, output_yaw[None]], axis=0)
    ctrl_action = ( raw_ctrl_action - env._ctrl_median[:4] ) / env._ctrl_halfspan[:4]
    
    select_action = ( ( ( 2 * cube_id + 1) * jnp.pi / 3 ) - jnp.pi ) / ( jnp.pi )

    action =  jnp.concatenate([ctrl_action, select_action[None]], axis=0)
    action = jnp.clip(action, -1, 1)
    
    return action

In [160]:
cube_id  = 0
cube_mass = 0.07936
gravity = 9.81
gravity_comp = cube_mass * gravity

In [162]:
camera = mujoco.MjvCamera()
camera.distance = 0.8
camera.lookat = np.array([0.4, 0.0 , 0.4])
camera.elevation = -30.0
camera.azimuth = 180

In [163]:
rollout = []

env_state = reset_fn(key_env)
rollout.append(env_state)

for i in range(default_config.episode_length):
    wp = get_waypoint(env_state)
    action = get_action(env_state, wp, cube_id)
        
    env_state = step_fn(env_state, action)
    rollout.append(env_state)

In [164]:
video_images = []
mocap_key = 'target_mocap'
for i in range(default_config.episode_length):
    if i % 2 == 0:
        video_images.append(
            env.render_from_info(
                rollout[i].data.qpos,
                rollout[i].data.qvel, 
                rollout[i].info[f'{mocap_key}_pos'],
                rollout[i].info[f'{mocap_key}_quat'],
                camera=camera,
            )
        )

In [165]:
media.show_video(video_images, fps=1.0 / env.dt / 2)

This browser does not support the video tag.
